# Dynamic STS/RTR State Analysis on DoC AAL90 BOLD

This notebook implements a first dynamic local-PhiID analysis using the local atom output `L` from `PhiIDFull`.

Core idea:
- for each ROI pair, extract local `L.sts(t)` and `L.rtr(t)`
- build time-resolved edgewise STS/RTR feature vectors
- cluster pooled dynamic features across subjects and conditions
- compare reconstructed state centroids to FC and SC

## 1. Prepare local PhiID extraction

The CLI helper writes the exact MATLAB batch command for local STS/RTR extraction:

```bash
python /Users/borjan/CNRS/projects/TVBToolkit/scripts/run_empirical_bold_local_phiid.py \
  --redundancy mmi
```

The MATLAB runner to compute local edge time series is:
`/Users/borjan/CNRS/projects/TVBToolkit/scripts/phiid_empirical_bold_local_sts_rtr_aal90.m`

Output files are one subject each:
`<subject_stub>__local_sts_rtr_mmi.mat`

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/Users/borjan/CNRS/projects/TVBToolkit')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from tvbtoolkit.analysis.dynamic_phiid import (
    load_local_phiid_index,
    fit_incremental_pca,
    pool_reduced_dynamic_features,
    score_kmeans_range,
    fit_dynamic_state_model,
    reconstruct_state_centroids,
    compare_centroids_to_connectomes,
    connectivity_vectors_from_records,
)
from brain_states_new_doc_bold_audited import load_new_doc_subjects, _maybe_apply_roi_reordering


In [ ]:
OUT = ROOT / 'results' / 'phiid_empirical_bold_dynamic'
LOCAL_DIR = OUT / 'local_phiid' / 'mmi'
MANIFEST = OUT / 'inputs' / 'manifest.csv'

index_df = load_local_phiid_index(LOCAL_DIR, manifest_path=MANIFEST)
index_df.head()

## 2. Recommended dynamic feature choice

Use `mode='balance_and_magnitude'` as the main model.

- `balance = z(STS) - z(RTR)`
- `magnitude = z(STS) + z(RTR)`

This keeps the dynamic synergy-vs-redundancy contrast in one shared state space.

In [ ]:
pca_model = fit_incremental_pca(
    index_df,
    mode='balance_and_magnitude',
    n_components=32,
    smooth_window=3,
    zscore_within_subject=True,
)

reduced_x, time_df, edge_meta = pool_reduced_dynamic_features(
    index_df,
    pca_model=pca_model,
    mode='balance_and_magnitude',
    smooth_window=3,
    zscore_within_subject=True,
)

print(reduced_x.shape)
time_df.head()

In [ ]:
k_scan = score_kmeans_range(reduced_x, k_values=range(3, 11), random_state=7)
k_scan

## 3. Fit pooled dynamic states

Pick `k` using stability, silhouette, and downstream interpretability, not just one metric.

In [ ]:
k = 5
state_model, state_labels = fit_dynamic_state_model(reduced_x, k=k, random_state=7)
time_df['state'] = state_labels
time_df.head()

In [ ]:
centroids = reconstruct_state_centroids(
    pca_model,
    state_model,
    mode='balance_and_magnitude',
    edge_i=edge_meta['edge_i'],
    edge_j=edge_meta['edge_j'],
    n_regions=90,
)
centroids.keys()

## 4. Compare state centroids to subject FC and SC

This is the simplest first comparison:
- reconstructed STS centroid vs subject FC / SC
- reconstructed RTR centroid vs subject FC / SC

In [ ]:
records, _ = load_new_doc_subjects(ROOT / 'data' / 'doc_patients_new_data')
records_use, _, _ = _maybe_apply_roi_reordering(records, mode='aal90_fc')
connectomes = connectivity_vectors_from_records(records_use)
comparison_df = compare_centroids_to_connectomes(centroids, connectomes)
comparison_df.head()